<a href="https://colab.research.google.com/github/Heba-Ajami/syria-parliament-elections-2025/blob/main/Syrian_Parliament_Election_2025_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PROJECT: Building a Clean Voter + Candidate Master Dataset from PDFs

In [ ]:

!pip install pdfplumber arabic-reshaper python-bidi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 kB 5.9 MB/s eta 0:00:00


In [ ]:
import re
import pdfplumber
import pandas as pd
import arabic_reshaper
from bidi.algorithm import get_display
import os

# ---------- Helper functions ----------
def fix_arabic(text):
    """Fix Arabic text shaping + direction"""
    if not text:
        return ""
    reshaped = arabic_reshaper.reshape(text.strip())
    return get_display(reshaped)

You will upload all 25 PDF files here:
- 12 first lists  (e.g. damascus_first.pdf, aleppo_first.pdf, ...)
- 11 final lists  (e.g. damascus_final.pdf, ...)
- 1 candidates list (e.g. candidates.pdf)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Aleppo - Final.pdf to Aleppo - Final.pdf


In [ ]:
pdf_path = "Aleppo - Final.pdf"   # Upload your test file with this exact name in Colab
rows = []
current_district = ""


with pdfplumber.open('Aleppo - Final.pdf') as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if not text:
            continue

        for line in text.split("\n"):
            line = line.strip()

            # Detect district line and normalize phrasing
            if "في الانتخابیة الدائرة" in line or "في الانتخابية الدائرة" in line:
                normalized = line.replace("في الانتخابیة الدائرة", "الدائرة الانتخابية في")
                normalized = normalized.replace("في الانتخابية الدائرة", "الدائرة الانتخابية في")
                current_district = fix_arabic(normalized)
                continue

            # Detect row pattern: number | name | birth year / region
            match = re.match(r"^(\d+)\s+(.+?)\s+(\d{4})\s*/\s*(.+)$", line)
            if match:
                no = match.group(1)
                name = fix_arabic(match.group(2))
                birth_year = match.group(3)
                region = fix_arabic(match.group(4))

                rows.append({
                    "City": city,
                    "ListType": list_type,
                    "District": current_district,
                    "No": no,
                    "Name": name,
                    "BirthYear": birth_year,
                    "Region": region
                })

# ---------- Save to CSV ----------
df = pd.DataFrame(rows)
csv_filename = f"{city}_{list_type}.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8-sig")

print(f"✅ Extracted {len(df)} rows. CSV saved as {csv_filename}.")
df.head(10)

NameError: name 'city' is not defined

STEP 3. Define Helper Extraction Function
----------------------------------------
Each city-PDF contains multiple pages → each page is a "table".
- The *first cell* (row 0, col 0) of each table = electoral district.
- Then we have 2 columns: "Name" and "Birthplace + Year".


In [ ]:
def extract_from_text(pdf_file, city, list_flag):
    records = []

    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            header_text = page.extract_text() or ""
            district_match = re.search(r'الدائرة الانتخابية في (.+)', header_text)
            district = district_match.group(1).strip() if district_match else ""

            lines = (page.extract_text() or "").split("\n")

            for line in lines:
                if "الاسم" in line or "المواليد" in line:
                    continue

                # 1) with birthplace + slash + year
                m = re.match(r"^(\d+)\s+(.+?)\s+(.+?)\s*/\s*(\d{4})$", line.strip())
                # 2) with birthplace + year (no slash)
                if not m:
                    m = re.match(r"^(\d+)\s+(.+?)\s+(.+?)\s+(\d{4})$", line.strip())
                # 3) with name + year only (no birthplace)
                if not m:
                    m = re.match(r"^(\d+)\s+(.+?)\s*/\s*(\d{4})$", line.strip())

                if m:
                    groups = m.groups()
                    serial = groups[0]
                    name = groups[1]
                    birthplace = groups[2] if len(groups) > 3 or len(groups) == 4 else ""
                    year = groups[-1]

                    records.append({
                        "name": name.strip(),
                        "city": city,
                        "district": district,
                        "birthplace": birthplace.strip(),
                        "birth_year": year,
                        list_flag: 1
                    })
    return pd.DataFrame(records)

STEP 4. Process All Uploaded PDFs
For each file:
- detect list type (first / final / candidate) from its name
- get city (from filename prefix, except candidates)
- extract all tables + districts

In [ ]:
all_dfs = []

# Loop through uploaded files
for filename in uploaded.keys():
    fname = filename.lower()

    if "first" in fname:
        list_flag = "in_first"
    elif "final" in fname:
        list_flag = "in_final"
    elif "candidate" in fname:
        list_flag = "is_candidate"
    else:
        list_flag = "unknown"

    # use city name only (strip extension & flags)
    base = fname.replace(".pdf","")
    city = base.split("-")[0].strip()

    df = extract_from_text(filename, city, list_flag)
    all_dfs.append(df)

# Combine into one full dataframe
combined = pd.concat(all_dfs, ignore_index=True)

# Preview first rows
combined.head(20)


""


 STEP 5. Merge Data to Master Table
-People appear in multiple lists (first, final, candidate).
-We merge them on name+city+district+birthplace+birth_year
-and give each one flags (1/0).

In [ ]:
master = combined.groupby(
    ["name","city","district","birthplace","birth_year"],
    as_index=False
).agg({
    "in_first": "max",
    "in_final": "max",
    "is_candidate": "max"
})

# Ensure flag columns exist and fill NaN with 0
for col in ["in_first","in_final","is_candidate"]:
    if col not in master:
        master[col] = 0
master = master.fillna(0)

master.head(20)

STEP 6. Save and Download Clean Dataset
Save everything into a UTF-8 CSV (handles Arabic names if present).


In [ ]:
master.to_csv("clean_master.csv", index=False, encoding="utf-8-sig")

from google.colab import files
files.download("clean_master.csv")